In [1]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=SQvm5Bm41RJg7wCTMevL62rWFWDK5u&access_type=offline&code_challenge=l5Cf2QRNmOCr9CLm7BAwo6MB9ZNk-mMk2hRgZY8dZhM&code_challenge_method=S256


Credentials saved to file: [/Users/meghakaladharreddypothamsetty/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "zprocure" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [2]:
import asyncio
from google import genai
from google.genai import types
import os
# ---------- Setup ----------

client = genai.Client(
    vertexai=True,
    project="aistimate",
    location="global",
)

model_name = "gemini-2.5-pro"

generate_content_config = types.GenerateContentConfig(
    temperature=0,
    top_p=1,
    seed=7,
    max_output_tokens=65535,
    safety_settings=[
        types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="OFF"),
    ],
    thinking_config=types.ThinkingConfig(thinking_budget=-1),
)

# ---------- Helper ----------

def make_part(path: str) -> types.Part:
    with open(path, "rb") as f:
        data = f.read()
    ext = path.split(".")[-1].lower()
    mime = {
        "pdf": "application/pdf",
        "png": "image/png",
        "jpg": "image/jpeg",
        "jpeg": "image/jpeg",
        "txt": "text/plain",
        "json": "application/json"
    }.get(ext, "application/octet-stream")
    return types.Part.from_bytes(data=data, mime_type=mime)
# ---------- Output Saving ----------







In [23]:
def build_prompts():
        return {
    "CODE_LOOKUP": """You are a building code compliance analyst preparing an expert report from the attached carrier estimate or supplemental documents.

OBJECTIVE:
Extract location-specific property information and generate a code compliance matrix to assess the validity of the restoration scope. Your matrix must allow technical, legal, and enforcement-level review.

PART 1: PROPERTY IDENTIFICATION

Extract and display the following details explicitly in the below format (all values below are examples):

| Field | Value | Source |
|-------|-------|--------|
| Full Street Address | [123 Main St, Anytown, TX 75001] | [Page #, Document Name] |
| Municipality or Jurisdiction | [City of Anytown] | [zoning info, doc line #] |
| County | [Dallas County] | [tax record or doc] |
| Incorporated/Unincorporated | [Incorporated] | [jurisdictional map or site] |
| ZIP Code | [75001] | [carrier estimate] |
| Inspection/Report Date | [March 15, 2024] | [carrier estimate or inspection] |

MUST include source citations in all rows.

PART 2: CODE STACK DETERMINATION

Using the jurisdiction and inspection date, determine the full set of codes applicable at the time of inspection. List each code family separately in the below format (all values below are examples):

| Code Type | Version/Edition | Citation Source | Applied Amendments | Enforceability Level |
|-----------|------------------|------------------|---------------------|------------------------|
| IRC | 2018 IRC | [ICC database, city site] | [NCTCOG mods] | Mandatory |
| NEC | 2023 NEC | [NEC.gov] | [none] | Mandatory |
| IECC | 2021 IECC | [DOE/state site] | [state energy code mods] | Mandatory |

For each code:
- Confirm it applies to residential work
- Cross-verify adoption at municipal, county, and state levels
- Include any stricter local amendments or enforcement practices

Do not skip any level. If any level lacks information, state "Unknown" and flag it.

PART 3: CODE COMPLIANCE MATRIX

Build a matrix of building components and related code requirements, grounded in enforceable citations and carrier estimate inclusion review.

Matrix Format(all values below are examples):

| Affected System | Code Section | Code Summary | Interpretation | Required By Code | Present in Carrier Estimate | Justification |
|------------------|---------------|----------------|------------------|-------------------|-------------------------------|----------------|
| Roofing | IRC R905.2.8.5 | Drip edge required at eaves & rakes | Must be installed per manufacturer & code | Yes | No | Missing on eaves; required due to full shingle tear-off |

Mandatory Coverage Areas (review all):
- Roofing: decking inspection, underlayment, ice/water shield, flashing, drip edge, ventilation, fasteners
- Siding: WRB, flashing, trim
- Windows: flashing, insulation, support
- Framing: blocking, shear, load path
- Electrical: grounding, junctions, disconnects
- HVAC: clearances, lines, platforms, ductwork

For each system:
- Include at least one row per bullet above unless clearly not applicable
- Use code section numbers (e.g., IRC R806.2)
- Justify "Not Required" with exact wording from code or trigger logic
- List at least one field as "No" under 'Present in Carrier Estimate' for auditing

VALIDATION AND QUALITY CHECKS

Code Version Accuracy:
- Confirm all adopted code versions match inspection/report date

Trigger Identification:
- Every cited requirement must identify the action that triggers it (e.g., roof tear-off)

Estimate Comparison:
- Flag all omissions; justify any marked "Yes" with estimate page or line reference

"N/A" Handling:
- Only use "Not Applicable" when justified by jurisdictional exemption or clear logic

Jurisdiction Checks:
- Verify amendments using official municipal, county, and state websites
- Cite the source used for each code adoption decision

Matrix Completeness:
- If a system has no entries, explain why
- Include "Inspection Required" for components not visibly verifiable

DAUBERT RELIABILITY REQUIREMENTS

Your analysis must meet legal admissibility standards:

- Reliability: All citations must be verified through ICC, NEC, or government sources
- Known Error Rate: Flag interpretations that involve ambiguity or enforcement discretion
- Peer Review: Reference ICC, AIA, NCARB or equivalent professional publications
- General Acceptance: Ensure all recommendations reflect standard industry enforcement

FINAL OUTPUT FORMAT

Provide the following 3 sections in order:

1. Property Details Table
2. Code Adoption Table
3. Code Compliance Matrix Table

Use markdown-compatible table formatting if supported. Do not summarize or paraphrase—this is a structured code compliance report.""",

    "REPORT_ANALYSIS": """You are a forensic damage analyst specializing in residential and commercial property insurance claims. You are provided with one or more of the following document types as attachments:

* Forensic inspection reports (e.g., roof, interior, structural)
* Annotated images or photo sets
* Engineering letters or reports
* Aerial roof measurement data (e.g., EagleView or similar)
* Claim summaries, contractor notes, and carrier communications
* Moisture readings, thermal imaging, or environmental testing data
* Plaintiff estimates, contractor scopes, or settlement documentation

NOTE: The file names may vary and may not explicitly state their contents. Do not rely on filenames. Instead, identify each document type by its internal content.

OBJECTIVE:
Perform a complete, evidence-based analysis of all observable damages by room or elevation. For every observed condition, extract supporting documentation and identify what restoration work is needed. **DO NOT QUANTIFY OR MEASURE** - that will be handled in the estimation stage.

DO NOT SKIP ANY STEP IF A DOCUMENT TYPE IS MISSING. Use only what is available and note omissions where appropriate. Proceed regardless of gaps in input.

STRUCTURE:
Repeat the following format for **every room, elevation, or system area** with documented or inferable damage.

---

### [Room or Elevation Name]

DAMAGE DOCUMENTATION

* **Primary Damage**: Describe the main damage (e.g., water stain, blistering, rot, delamination)
* **Secondary Damage**: Any follow-on effects (e.g., mold, insulation compromise, trim swelling)
* **Evidence Sources**: Reference Photos [IDs or filenames], Report pages [#], Inspection Notes, and document source

AFFECTED COMPONENTS

List all building components that show damage or require restoration work:

* **Structural Elements**: (e.g., ceiling joists, wall framing, roof decking)
* **Finish Materials**: (e.g., drywall, paint, flooring, trim)
* **Systems**: (e.g., electrical fixtures, HVAC components, plumbing)
* **Insulation/Barriers**: (e.g., insulation, vapor barriers, house wrap)

CARRIER ESTIMATE COMPARISON

* **Included Scope**: List what the carrier did include (line item description)
* **Missing Scope**: Items observed but omitted in carrier scope
* **Justification for Inclusion**: Why the omitted item is required (trigger logic, industry standard, interdependent component)

SYSTEM INTEGRATION IMPACTS

* **Code Triggers**: Any work that initiates building code compliance requirements (e.g., R-value updates, decking inspections)
* **Aesthetic Impacts**: Line-of-sight disruptions, matching failures (color, texture, material)
* **Sequence Dependencies**: Where partial work is infeasible due to construction sequence (e.g., finish ceiling must follow after insulation)

MOISTURE ASSESSMENT (If Applicable)

* **Moisture Evidence**: Document any moisture readings, thermal imaging findings, or water infiltration evidence
* **Drying Requirements**: Note need for thermal imaging, hygrometer readings, and drying equipment per IICRC S500 protocols
* **Monitoring Protocol**: Specify continuous moisture monitoring with data loggers during drying process
* **Documentation**: Note requirement for drying logs and progress tracking to mitigate mold risk

RISK ASSESSMENT

* **Immediate Concerns**: Mold, electrical hazard, structural weakness, open exposure
* **Long-Term Implications**: Warranty issues, insulation failure, degraded performance, continued leakage
* **Mitigation Requirements**: Any required steps to prevent escalation (e.g., full replacement, drying protocol, temporary protection)

---

ADDITIONAL SCOPE REQUIREMENTS
For each damaged area, also evaluate and include where applicable:

SITE TESTING & INSPECTIONS (Include Only If Evidence Supports)
- Water Infiltration Testing: ASTM E1105 window infiltration tests on representative units
- Structural Inspections: Plywood sheathing inspection per IRC R804.3 after tear-off (only if roof damage requires tear-off)
- Moisture Testing: Initial and ongoing moisture content readings (only if moisture intrusion is documented)
- Air Quality Testing: If mold risk is present, specify testing protocols (only if mold conditions are observed or suspected)

SITE PROTECTION & LOGISTICS
- Containment Systems: ZipWall barriers, HEPA negative-air units (identify need, not quantity)
- Waste Management: Dumpster staging and capacity requirements
- Professional Cleaning: Post-construction cleaning protocols
- Temporary Protection: Weather protection, security measures

PROJECT MANAGEMENT & COORDINATION
- Supervision Requirements: On-site management needs
- Permit Coordination: Municipal permitting and inspection attendance requirements
- Third-Party Coordination: Management of testing companies, inspectors, utilities

INTERIOR RESTORATION PROTOCOLS (Include Only If Damage Present)
- Stain-Blocking Primer: Required on all flood-cut walls and ceilings per PDCA guidelines before finish coats (only if water damage occurred)
- Moisture Barrier Installation: Where water infiltration occurred (only if infiltration is documented)
- Insulation Replacement: Full replacement when contaminated or compressed (only if insulation damage is observed)
- Trim Matching: Line-of-sight matching requirements across affected areas (only if trim damage is present)

ROOFING SYSTEM REQUIREMENTS
- Full System Replacement: When hail damage affects multiple components
- Decking Assessment: Complete sheathing inspection and replacement where compromised
- Ventilation Upgrades: Code-required ventilation improvements
- Ice & Water Shield: Enhanced protection in vulnerable areas

---

EVIDENCE-BASED ANALYSIS PRINCIPLES
CRITICAL: Only include scope items that are supported by documented evidence. If a particular type of damage is not present (e.g., no moisture damage, no window damage, no structural damage), explicitly state this rather than recommending unnecessary protocols.

Examples of Proper Evidence-Based Responses:
* "No moisture damage observed in available documentation - moisture mapping protocols not applicable"
* "No window damage documented - ASTM E1105 testing not required"
* "Roof damage limited to shingles only - sheathing inspection not triggered"
* "No interior water intrusion evident - stain-blocking primer not required"

EVIDENCE CORRELATION GUIDELINES

For **every damage condition**, you must:
* Link to at least one **photo ID** or annotated image or logical reasoning from analysis
* Cite page number or section from the relevant inspection, engineering, or aerial report
* If damage extent is inferred (e.g., from image context), specify the method used
* Do not make undocumented assumptions; flag any gaps explicitly

ALWAYS USE language like:
* "As shown in Photo 14, ceiling discoloration extends across the entry area"
* "Page 3 of Engineering Report notes compromised rafter tail at southwest corner"
* "Thermal imaging on page 7 shows moisture intrusion beyond visible staining"

MANDATORY OUTPUT REQUIREMENTS

1. You must output one full section per distinct room/elevation/system
2. Do not omit any damaged item or area, even small trim or ceiling tape lines
3. Tie every observed damage to at least one form of verifiable evidence
4. If no damage is observed in an area, write:
   "**No observable damage**: This room showed no visible damage based on available documents and photos. No scope required."
5. Include all ancillary requirements (testing, monitoring, protection, management)

VALIDATION PROTOCOLS

* Every damage claim must have at least one evidence reference
* All scope requirements must be traceable to damage or code triggers
* No item should be included without a supporting cause
* "Not observed" is acceptable only with explanation (e.g., area not visible in photos)
* Cross-reference all ancillary requirements with applicable codes and standards

QUALITY ASSURANCE CHECKLIST

* Verify all room/elevation names match across report and photo metadata
* Cross-check all photo references are unique and traceable
* Validate that all testing, monitoring, and protection requirements are included
* Confirm project management and coordination needs are addressed

ALLOWED ASSUMPTIONS

If any document type is missing:
* Use remaining evidence to fill gaps
* Infer plausible scope based on visible consequences
* Flag assumptions clearly as "inferred from photo only" or "no direct engineering note found"
* Apply industry standards for testing and monitoring when specific protocols aren't documented

You must **never halt or skip analysis due to missing files**.

CONCLUSION

Complete the output for all rooms or elevations with observed damage. Do not stop at major rooms—include small closets, utility areas, and transitions. Every component matters, including all necessary testing, monitoring, site protection, and project management requirements for complete restoration.""",


"SCOPING_LOGIC": """You are a restoration scoping expert. Your task is to identify exactly which scopes of work are required to fully repair all documented damages, meet all code requirements, and satisfy proper sequencing and standard construction logic.

---

### INPUTS

You are provided the following:

1. **Code Lookup Output** – detailing building code triggers (e.g., IRC, NEC) based on location, inspection date, and system type.
2. **Damage Report Analysis** – a room-by-room breakdown of damages, systems affected, carrier estimate omissions, and risk factors.
3. **Scope Catalog** – a list of available scopes, each with:
   - Scope Title
   - Scope ID
   - Unit Type (SF, LF, EA, etc.)

---

### OBJECTIVE

Generate a list of scope IDs and corresponding quantities required to restore the structure in compliance with:

- All enforceable building codes
- Damage observations
- Aesthetic matching and LKQ standards
- Construction sequencing (demo → rough → finish)
- Industry-required site protection and general conditions

Only select from scopes available in the scope catalog (by `scope_id`), and ensure the `quantity` you provide uses the correct unit type associated with each scope.

---

### OUTPUT FORMAT (strictly required)

Return a **JSON array**, where each item includes:
[
  {
    "scope_id": "abc12345",
    "quantity": 250,
    "justification": "Exterior trim observed to be water-damaged and swollen on all elevations; full replacement required per matching standards and photos 12–14."
  },
  {
    "scope_id": "def67890",
    "quantity": 42,
    "justification": "Roof tear-off triggers IRC R806.2; ridge vent must be installed for code-compliant ventilation system."
  }
]""",

 "ESTIMATE" : """You are a certified insurance restoration estimator and compliance analyst. You are provided with:

1. The final list of selected scopes (scope_id, title, unit, quantity)
2. Line-item descriptions and pricing range from Clear Estimates (CE)
3. Code Lookup results (e.g., IRC, NEC, IECC)
4. Report Analysis findings (damage detail, photo refs, inspection reports)

---

## OBJECTIVE

Rather than generating new estimates, your task is to **justify and explain** the inclusion of each line item using evidence-based reasoning and code triggers.

For each line item, you will:

1. **Retain the standard Xactimate markdown table format**.
2. **Fill in all fields** using the information from scope selection, CE description, and available inputs.
3. **After the table**, provide a structured justification for each line item.
4. **If any required line item is missing**, include it in an **Addendum** with justification and citation.

Do NOT calculate pricing. Instead, **use the existing cost values provided** in the scope data.
Do not extra items. Use the ones provided in the scope selection. Also use the lower ppricing values in the range in the scope selection

---

## REQUIRED OUTPUT FORMAT

Repeat the following format for each system or elevation.

### **[System or Elevation Name]**

```markdown
| CAT | SEL | DESCRIPTION | QTY | UNIT | UNIT PRICE | TAX | O&P | RCV | DEPREC. | ACV | SOURCE |
|-----|-----|-------------|-----|------|------------|-----|-----|-----|----------|-----|--------|
| ROF | RFG240 | R&R Dimensional 30yr Shingles | 2800 | SF | $8.12 | $0.00 | $0.00 | $22,734.37 | $0.00 | $22,734.37 | CE Scope a84dc25a...
```
Line-by-Line Justification:

Scope Title: Roofing Replacement – Dimensional

Reason for Inclusion: Full replacement required due to hail damage and uniform degradation; observed in Photos 3–5.

Code Trigger: IRC R908.3.1 – "Existing roof coverings shall not be applied over water-soaked or deteriorated materials."

Source of Quantity: EagleView aerial sketch; confirmed by inspection report (pg 4).

Matching Standard: Full replacement required per aesthetic uniformity and manufacturer requirements.

Carrier Estimate Variance: Carrier omitted drip edge and ventilation, violating IRC R905.2.8.5 and R806.2.




GRAND TOTALS


```markdown

### **GRAND TOTALS**

| Category | Subtotal | Source |
|----------|----------|--------|
| Roofing & Gutters | $29,870.45 | IRC R905, R806 / CE scope IDs a84dc25a, ceedede7, etc. |
| General Demolition | $2,278.12 | CE Scopes a84dc25a, e0887d47 |
| Exterior Trim | $92.82 | CE Scope 196db049 |
| Project Finalization | $1,680.00 | CE Scope a84dc25a |
| **Grand Total (RCV)** | **$33,921.39** | |
```
VALIDATION CHECKLIST
All line items in provided scope must appear in the estimate section

Every line includes valid code/damage justification

Quantities match scope

Pricing exactly matches input

Addendum only includes missing, required, justifiable scope

No math performed — totals directly taken from input

Grand Total (RCV) reflects the total of justified + added items

Every item justified as code-required, damage-based, or matching-triggered

This justification report will be used in legal and appraisal settings. Do not skip or assume.
""",


    "DAUBERT_ESTIMATE_OUTPUT": """
You are a forensic Daubert-compliant expert witness preparing a final "Plaintiff-style" Xactimate estimate report ready for legal submission.

OBJECTIVES:
1. Synthesize all upstream analyses into one cohesive document.
2. Mirror the layout, level of detail, and citation style found in the primary reference document provided (Grace Forensic Loss Consultants, April 2025):
   - Cover page with case header (Insured, Claim #, Property, Dates)
   - Table of Contents
   - Line-item tables by area (roof, exterior elevations, general conditions, etc.) in Xactimate format with CAT/SEL codes
   - Summaries (by elevation, by category, grand totals) with precise math
   - "Daubert Reliability" section noting sources, known error rates, peer-review references, and confirmation of code/version accuracy
   - Appendices for photos, code citation library, and evidence matrix

REQUIREMENTS:
- Use exact Xactimate table columns:

| CAT | SEL | DESCRIPTION | QTY | UNIT | UNIT PRICE | TAX | O&P | RCV | DEPREC. | ACV | SOURCE |

- Include a formal "Expert Opinion & Methodology" narrative conforming to Daubert standards
- All citations must reference either your prior stage ("[Stage] Output") or the reference document(s) provided
- Maintain legal-grade formality and ready-for-court structure

FINAL OUTPUT:
Produce a single json (or PDF-ready) document that a court could receive as the expert's estimate exhibit.""",

    "REBUTTAL": """You are a forensic rebuttal specialist responding to a deficient insurance carrier estimate. Your response must be formal, detailed, and based in code, evidence, and industry logic.

**OBJECTIVE**: Create comprehensive, defensible rebuttal documentation with legal and technical precision.

### **I. Summary of Discrepancies**
Categorize the major classes of omissions (e.g., code compliance, aesthetic mismatch, missing scope) with high-level bullets.

---

### **II. Room-by-Room Rebuttal**

#### [Room or Elevation Name]
- **Issue:** What was omitted or under-scoped
- **Evidence:** Photo X, Report pg Y
- **Code/Standard:** IRC section, IICRC standard, or Xactimate convention
- **Correct Scope:** Describe what should be included
- **Reasoning:** Include logic based on damage extent, mismatch, sequence of construction

---

### **III. Code Violations**
List every component omitted or under-scoped that violates building code in the below example format:
- IRC R908.3.1: Decking not allowed to remain without inspection
- NEC 820.100: Satellite system ungrounded

---

### **IV. General Conditions & O&P Justification**
- Number of trades
- Need for project supervision
- Dumpster/toilet/storage logic
- Code-permitted markup (O&P)

---

### **V. Aesthetic & Matching Justifications**
- Why patching fails LKQ standard
- Photo-based mismatch documentation
- Manufacturer unavailability (if applicable)

---

### **VI. Conclusion**
Summarize:
- # of omitted rooms or trades
- Major life-safety risks or code issues
- Estimated value delta (if known)
- Your demand: "We respectfully request that the omitted items be added and paid in full."

**VALIDATION CHECKLIST**:
- [ ] Every deficiency has supporting evidence
- [ ] All code citations are current and accurate
- [ ] Financial calculations are mathematically correct
- [ ] Professional tone maintained throughout
- [ ] Specific actions requested clearly stated
- [ ] Documentation references complete and accurate

Maintain a clear, professional tone rooted in documentation."""
        }

In [20]:
import json

def convert_scopes_to_llm_text(input_path="scopes.json", output_path="scopes_for_llm.txt"):
    try:
        with open(input_path, "r") as f:
            scopes = json.load(f)

        output_lines = ["### Available Scopes (from Clear Estimates API)", ""]

        for scope in scopes:
            title = scope.get("title", "Untitled")
            scope_id = scope.get("scope_id", "unknown")
            units = scope.get("units", "unspecified")

            # Concise bullet format
            output_lines.append(f"- **{title}**  \n  `scope_id: {scope_id}`  \n  _Units_: {units}")
            output_lines.append("")  # extra newline for spacing

        # Write to a .txt file
        with open(output_path, "w") as f:
            f.write("\n".join(output_lines))

        print(f"✅ {len(scopes)} scopes written to {output_path}")

    except Exception as e:
        print(f"❌ Error during scope conversion: {e}")


In [5]:
convert_scopes_to_llm_text()

✅ 322 scopes written to scopes_for_llm.txt


In [17]:
def load_llm_scopes_from_file(filepath):
    """Load LLM output from a text file containing JSON array."""
    try:
        with open(filepath, "r") as f:
            # Clean any stray markdown fencing or whitespace if needed
            raw = f.read().strip()
            # In case file is wrapped in ```json ... ```
            if raw.startswith("```json"):
                raw = raw.strip("```json").strip("```")
            data = json.loads(raw)
        print(f"✅ Loaded {len(data)} scope items from {filepath}")
        return data
    except Exception as e:
        print(f"❌ Failed to load file {filepath}: {e}")
        return []

def create_estimate_from_llm_output(llm_output, zipcode="78749", estimate_type=3):
    url = BASE_URL + "estimates/create"
    headers = {"x-api-key": API_KEY}

    estimate_array = [
        {"scope_id": item["scope_id"], "quantity": item["quantity"]}
        for item in llm_output
    ]

    payload = {
        "zipcode": zipcode,
        "estimatearray": estimate_array,
        "type": estimate_type
    }

    response = requests.post(url, headers=headers, json=payload)
    print("ESTIMATE REQUEST:", response.status_code)

    try:
        response_data = response.json()
        print(json.dumps(response_data, indent=2))
        return response_data
    except json.JSONDecodeError:
        print("Invalid JSON in response")
        return None


In [24]:
# ---------- Async Gemini Runner ----------

async def run_block(label, prompt, file_parts=None):
    contents = [types.Content(role="user", parts=[types.Part.from_text(text=prompt)])]
    if file_parts:
        contents[0].parts.extend(file_parts)

    output = ""
    try:
        print(f"🔹 Running {label}...")
        stream = client.models.generate_content_stream(
            model=model_name,
            contents=contents,
            config=generate_content_config
        )
        for chunk in stream:  # ✅ DO NOT use 'await'
            output += chunk.text
        print(f"✅ {label} complete ({len(output)} chars)")
    except Exception as e:
        output = f"[ERROR in {label}] {e}"
        print(output)

    return label, output




# ---------- Master Pipeline ----------

async def run_aistimate_pipeline(file_paths):
    prompts = build_prompts()

    # Assign files
    # 1) Carrier: only the first file
    carrier_parts = [make_part(file_paths[0])]

    # 2) Evidence: file_paths[0] plus file_paths[2:]
    evidence_paths = [file_paths[0]] + file_paths[2:]
    evidence_parts = [make_part(path) for path in evidence_paths]

    # 3) Policy: again, just the first file (if that’s what you meant)
    policy_parts = [make_part(path) for path in file_paths[1:2]]


    # Stage 1: Run code lookup & damage analysis in parallel
    stage1_tasks = [
        run_block("CODE_LOOKUP", prompts["CODE_LOOKUP"], carrier_parts),
        run_block("REPORT_ANALYSIS", prompts["REPORT_ANALYSIS"], evidence_parts),
    ]
    stage1_results = await asyncio.gather(*stage1_tasks)
    context = {label: output for label, output in stage1_results}

    for label, content in stage1_results:
        save_output(label, content)

    # Stage 2: Scoping logic (needs prior outputs)
    scoping_context = (
        f"--- CODE LOOKUP ---n{context['CODE_LOOKUP']}nn"
        f"--- DAMAGE OBSERVATIONS ---n{context['REPORT_ANALYSIS']}"
    )
    
    label, scoping_output = await run_block("SCOPING_LOGIC", prompts["SCOPING_LOGIC"] + "nn" + scoping_context,policy_parts)
    save_output(label, scoping_output)
    context["SCOPING_LOGIC"] = scoping_output
    filepath = "outputs/Jul25/2500095/run1/output_scoping_logic.txt"
    scope_data = load_llm_scopes_from_file(filepath)
    if scope_data:
    # Create estimate
        estimate_response = create_estimate_from_llm_output(scope_data, zipcode="80210", estimate_type=2)
    



# Use that structured version in the context
    estimate_context = (
    f"--- CODE MANDATES ---\n{context['CODE_LOOKUP']}\n\n"
    f"--- DAMAGE FINDINGS ---\n{context['REPORT_ANALYSIS']}\n\n"    
    f"--- SELECTED SCOPES ---\n{estimate_response}\n"
    )   

        # f"--- POLICY LOGIC ---n{context['POLICY_LOGIC_output']}nn"
    
    label, estimate_output = await run_block("ESTIMATE", prompts["ESTIMATE"] + "nn" + estimate_context)
    save_output(label, estimate_output)

    # Stage 4: Rebuttal
    # Stage 4: Rebuttal (pass carrier file for comparison)
    label, rebuttal_output = await run_block(
    "REBUTTAL",
    prompts["REBUTTAL"] + "nn" + estimate_output,
    file_parts=carrier_parts  # 🔹 passes carrier estimate as input context
    )
    save_output(label, rebuttal_output)


    return {
        "code_lookup": context["CODE_LOOKUP"],
        "report_analysis": context["REPORT_ANALYSIS"],
        "scoping_logic": context["SCOPING_LOGIC"],
        "estimate_output": estimate_output,
        "rebuttal_output": rebuttal_output,
    }


In [12]:
API_KEY = "emym6vnmxyo0d62vz3w1kaj"

# Step 2: Set the base URL (use sandbox for testing)
BASE_URL = "https://api.clearestimates.com/cepia/"

In [18]:
import requests
import json
filepath = "outputs/Jul25/2500095/run1/output_scoping_logic.txt"
scope_data = load_llm_scopes_from_file(filepath)
print("Loaded scopes:", scope_data)
if scope_data:
    # Create estimate
    estimate_response = create_estimate_from_llm_output(scope_data, zipcode="80210", estimate_type=2)

    if estimate_response:
        # Save to file
        with open("estimate.txt", "w") as f:
            f.write(json.dumps(estimate_response, indent=2))
        
    else:
        print("❌ Estimate creation returned no result.")
else:
    print("❌ No scope data loaded; cannot proceed with estimate creation.")



✅ Loaded 6 scope items from outputs/Jul25/2500095/run1/output_scoping_logic.txt
Loaded scopes: [{'scope_id': 'a84dc25a-485b-4f08-8470-8e95c943019d', 'quantity': 2800, 'justification': 'To address widespread, functional hail damage on dimensional shingles across all roof slopes, as documented in annotated photos. This scope covers the full tear-off and replacement of the roof system, including underlayment and starter strips, which is required for system integrity and to prevent future leaks.'}, {'scope_id': 'ceedede7-b7b0-44ac-892c-64da2e179538', 'quantity': 5, 'justification': 'Per IRC R905.1, roof sheathing must be inspected after tear-off. This is an allowance to replace any hail-fractured or delaminated decking panels discovered during inspection, ensuring a sound substrate for the new roof system.'}, {'scope_id': 'c4f9b3f8-6256-47dc-bc1f-baad1ea80d24', 'quantity': 100, 'justification': 'Code-mandated replacement of step flashing and valley flashing per IRC R903.2.1 and R905.2.8.2.

In [19]:
def save_output(label: str, content: str):
    output_dir = f"outputs/Jul25/2550002/run1"
    os.makedirs(output_dir, exist_ok=True)
        # run_name = run_index + 5
        # Define output file once for both stages
    filename = os.path.join(output_dir, f"output_{label.lower()}.txt")
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"📝 Saved: {filename}")
results = await run_aistimate_pipeline([
    "2550002/2550002 Carrier Estimate Insurance Carrier Estimate.pdf",
    "scopes_for_llm.txt",
    "2550002/2550002 Grace Forensic Damage Report and Photos Plaintiff Expert Estimate.pdf",
    "2550002/2550002_Expert_ReportsPhotosDiagrams_From_Fadner, Derek_Messenger_creation_3A97D8FE-84D6-4BFA-91AD-39F43E790DC5.jpeg",
    "2550002/2550002_Expert_ReportsPhotosDiagrams_From_Fadner, Derek_Messenger_creation_72CE907A-5EDA-42FC-A9C2-3D4245CC834E.jpeg",
    "2550002/2550002_Expert_ReportsPhotosDiagrams_From_Fadner, Derek_Messenger_creation_093D1853-BBE9-47A7-86EF-1769895C5FBE.jpeg",
    "2550002/2550002_Expert_ReportsPhotosDiagrams_From_Fadner, Derek_Messenger_creation_459D33C9-7EA8-451F-A4EE-1C09493F9BEA.jpeg"
])

FileNotFoundError: [Errno 2] No such file or directory: '2550002/2550002 Carrier Estimate Insurance Carrier Estimate.pdf'

In [26]:
def save_output(label: str, content: str):
    output_dir = f"outputs/Jul25/2500095/run3"
    os.makedirs(output_dir, exist_ok=True)
        # run_name = run_index + 5
        # Define output file once for both stages
    filename = os.path.join(output_dir, f"output_{label.lower()}.txt")
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"📝 Saved: {filename}")
results = await run_aistimate_pipeline([
    "Aiestimate/2500002_Ruiz Jr v. ASI Lloyds_2500005_Zimmerman v. Homesite Insurance Company_2500006_Zuniga v. Nati/117/ai estimate useful docs/2500095_Correspondence_Correspondence from IC_From_Nationwide Mutual Insurance Company_Supplement Estimate 2.pdf",
    "scopes_for_llm.txt",
    "Aiestimate/2500002_Ruiz Jr v. ASI Lloyds_2500005_Zimmerman v. Homesite Insurance Company_2500006_Zuniga v. Nati/117/ai estimate useful docs/Binder1 Photos_ Arnold, Marc.pdf",
    "Aiestimate/2500002_Ruiz Jr v. ASI Lloyds_2500005_Zimmerman v. Homesite Insurance Company_2500006_Zuniga v. Nati/117/ai estimate useful docs/LN Re-inspect Report.pdf"
])

🔹 Running CODE_LOOKUP...
✅ CODE_LOOKUP complete (7153 chars)
🔹 Running REPORT_ANALYSIS...
✅ REPORT_ANALYSIS complete (10870 chars)
📝 Saved: outputs/Jul25/2500095/run3/output_code_lookup.txt
📝 Saved: outputs/Jul25/2500095/run3/output_report_analysis.txt
🔹 Running SCOPING_LOGIC...
✅ SCOPING_LOGIC complete (2784 chars)
📝 Saved: outputs/Jul25/2500095/run3/output_scoping_logic.txt
✅ Loaded 7 scope items from outputs/Jul25/2500095/run1/output_scoping_logic.txt
ESTIMATE REQUEST: 200
{
  "estimate_id": 3385,
  "estimates": [
    {
      "title": "Roofing Replacement  - Dimensional",
      "description": "Demolish existing layer of roofing and replace with architectural/laminate/dimensional 240lb, 30 year fiberglass shingles. Flash chimney, plumbing vents, and valleys with aluminum. Continuous ridge vent. Additional features like dormers and skylights will add to the cost. Based on the square footage of the home's footprint.",
      "range": {
        "high": 31059.13,
        "low": 26725.29
 